# Полный анализ временных рядов
**От стационарности до поиска tipping points и фрактального анализа**

Автор: Grok (для тебя)

Используем два датасета:
- `daily-total-female-births.csv` — стационарный однородный ряд
- Nile River — неоднородный ряд второго рода (с regime shift)

In [ ]:
# === 1. УСТАНОВКА И ИМПОРТ БИБЛИОТЕК ===
!pip install pandas numpy matplotlib seaborn statsmodels pmdarima ruptures hurst -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.graphics.gofplots import qqplot
from pmdarima import auto_arima
import ruptures as rpt
from hurst import compute_Hc
import warnings
warnings.filterwarnings('ignore')

sns.set_style('darkgrid')
%matplotlib inline

## 2. Загрузка данных

In [ ]:
# Стационарный ряд
url_births = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-total-female-births.csv"
births = pd.read_csv(url_births, parse_dates=['Date'], index_col='Date')
print('Daily Total Female Births shape:', births.shape)

# Неоднородный ряд (Nile)
from statsmodels.datasets import nile
nile_data = nile.load_pandas().data
nile_data.set_index('year', inplace=True)
print('Nile River shape:', nile_data.shape)

## 3. Визуализация рядов

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 8))
births.plot(ax=ax[0], title='Стационарный ряд — Daily Female Births')
nile_data.plot(ax=ax[1], title='Неоднородный ряд — Nile River (второй род)')
plt.tight_layout()
plt.show()

## 4. Проверка на стационарность

In [ ]:
def test_stationarity(series, title=''):
    print(f'\n=== Тесты для {title} ===')
    # ADF
    adf = adfuller(series)
    print(f'ADF Statistic: {adf[0]:.4f} | p-value: {adf[1]:.4f}')
    # KPSS
    kpss_stat = kpss(series, regression='c')
    print(f'KPSS Statistic: {kpss_stat[0]:.4f} | p-value: {kpss_stat[1]:.4f}')

test_stationarity(births['Births'], 'Female Births')
test_stationarity(nile_data['volume'], 'Nile River')

## 5. Декомпозиция

In [ ]:
decomp = seasonal_decompose(births['Births'], model='additive', period=7)
decomp.plot()
plt.suptitle('Декомпозиция стационарного ряда', y=1.02)
plt.show()

## 6. ACF / PACF и QQ-plot

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
plot_acf(births['Births'], ax=axes[0,0], title='ACF')
plot_pacf(births['Births'], ax=axes[0,1], title='PACF')
qqplot(births['Births'], line='s', ax=axes[1,0])
axes[1,1].hist(births['Births'], bins=30)
axes[1,1].set_title('Гистограмма')
plt.tight_layout()
plt.show()

## 7. Модели ARMA / ARIMA / SARIMA

In [ ]:
# Автоматический подбор
model = auto_arima(births['Births'], seasonal=True, m=7, stepwise=True, trace=True)
print(model.summary())

## 8. Фрактальный анализ (Hurst exponent)

In [ ]:
def hurst_analysis(series):
    H, c, data = compute_Hc(series, kind='price', simplified=True)
    print(f'Hurst exponent = {H:.4f}')
    return H

print('=== Female Births (стационарный) ===')
hurst_analysis(births['Births'])
print('\n=== Nile River (неоднородный) ===')
hurst_analysis(nile_data['volume'])

## 9. Поиск tipping points (Change Point Detection)

In [ ]:
algo = rpt.Pelt(model='l2').fit(nile_data['volume'].values)
result = algo.predict(pen=20)
rpt.display(nile_data['volume'].values, result)
plt.title('Change Point Detection — Nile River')
plt.show()

## 10. Early Warning Signals (скользящие окна)

In [ ]:
def rolling_ews(series, window=30):
    roll_var = series.rolling(window).var()
    roll_acf = series.rolling(window).apply(lambda x: pd.Series(x).autocorr(lag=1))
    plt.figure(figsize=(12,4))
    plt.plot(roll_var, label='Rolling Variance')
    plt.plot(roll_acf, label='Rolling AR(1)')
    plt.legend()
    plt.title('Early Warning Signals')
    plt.show()

rolling_ews(nile_data['volume'])

## Выводы
В этом ноутбуке ты увидел полный цикл анализа временных рядов — от проверки стационарности до обнаружения regime shifts и фрактального анализа.